# Capítulo 2 – Working with Text Data: exercícios

*Build a Large Language Model (From Scratch)* – Sebastian Raschka

- **Exercício 2.1** – Byte pair encoding de palavras desconhecidas
- **Exercício 2.2** – Data loaders com strides e context sizes diferentes

In [ ]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

## Exercício 2.1 – Byte pair encoding de palavras desconhecidas


In [2]:
tokenizer = tiktoken.get_encoding("gpt2")

text = "Akwirw ier"
token_ids = tokenizer.encode(text)
print("Token IDs:", token_ids)

Token IDs: [33901, 86, 343, 86, 220, 959]


Decodificando cada token ID individualmente (mapeamento da figura 2.11):

In [3]:
for token_id in token_ids:
    print(f"{token_id:>6} -> {tokenizer.decode([token_id])!r}")

 33901 -> 'Ak'
    86 -> 'w'
   343 -> 'ir'
    86 -> 'w'
   220 -> ' '
   959 -> 'ier'


**Conclusão:** o BPE não precisa de um token `<|unk|>`. Palavras desconhecidas são quebradas em subpalavras e caracteres que estão no vocabulário, e por isso o `decode` reconstrói o texto original exatamente.

## Exercício 2.2 – Data loaders com strides e context sizes diferentes

Primeiro, o código do livro: download do texto, `GPTDatasetV1` (listing 2.5) e `create_dataloader_v1` (listing 2.6).

In [5]:
import os
import urllib.request

file_path = "the-verdict.txt"
if not os.path.exists(file_path):
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
    urllib.request.urlretrieve(url, file_path)

with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total de caracteres:", len(raw_text))

Total de caracteres: 20479


In [ ]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

Exemplo do livro como referência (`max_length=4`, `stride=1`):

In [7]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
print(next(data_iter))
print(next(data_iter))

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


### Configuração A: `max_length=2`, `stride=2`

In [8]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=4, max_length=2, stride=2, shuffle=False)
inputs, targets = next(iter(dataloader))
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)
print("\nNúmero de amostras no dataset:", len(dataloader.dataset))

Inputs:
 

tensor([[  40,  367],
        [2885, 1464],
        [1807, 3619],
        [ 402,  271]])

Targets:
 tensor([[  367,  2885],
        [ 1464,  1807],
        [ 3619,   402],
        [  271, 10899]])

Número de amostras no dataset: 2572


### Configuração B: `max_length=8`, `stride=2`

In [9]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=4, max_length=8, stride=2, shuffle=False)
inputs, targets = next(iter(dataloader))
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)
print("\nNúmero de amostras no dataset:", len(dataloader.dataset))

Inputs:
 tensor([[   40,   367,  2885,  1464,  1807,  3619,   402,   271],
        [ 2885,  1464,  1807,  3619,   402,   271, 10899,  2138],
        [ 1807,  3619,   402,   271, 10899,  2138,   257,  7026],
        [  402,   271, 10899,  2138,   257,  7026, 15632,   438]])

Targets:
 tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899],
        [ 1464,  1807,  3619,   402,   271, 10899,  2138,   257],
        [ 3619,   402,   271, 10899,  2138,   257,  7026, 15632],
        [  271, 10899,  2138,   257,  7026, 15632,   438,  2016]])

Número de amostras no dataset: 2569


**Conclusão:**

- `max_length` define quantos tokens cada linha (contexto) tem; os targets são os mesmos tokens deslocados uma posição para a direita.
- `stride` define quantas posições a janela anda entre uma amostra e a próxima.
- Com `max_length=2, stride=2` (stride = max_length) as amostras **não se sobrepõem**: cada token aparece em uma única linha de input.
- Com `max_length=8, stride=2` (stride < max_length) as amostras **se sobrepõem** em 6 tokens: há mais amostras, mas elas são muito redundantes, o que pode favorecer overfitting.